<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 컴퓨터 비전 · 06. Faster R-CNN

## Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks

- **원 논문:** [Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks](https://arxiv.org/abs/1506.01497)
- **저자 / 발표:** Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun · NeurIPS 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** RPN anchor, IoU assignment, objectness+box multi-task loss를 재현한다.

**축소하거나 생략한 부분:** VGG-16, 9 anchors/location, VOC/COCO 대신 4×4 단일-scale anchor grid를 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Fig. 3: sliding RPN | Task 1 anchor grid | 16 anchors |
| §3.1.1: anchors | Task 1 IoU | best anchor |
| §3.1.2 Eq. (1): Lcls+λLreg | Task 2 | finite loss |
| §3.1.2 Eq. (2): box parameterization | Task 1 delta | identity delta |

## 핵심 아이디어와 수식

$$L=\frac{1}{N_{cls}}\sum_iL_{cls}(p_i,p_i^*)+\lambda\frac{1}{N_{reg}}\sum_i p_i^*L_{reg}(t_i,t_i^*)$$

**기호 해설:** p는 objectness, p*는 anchor label, t는 box delta, λ는 regression 가중치다.

**코드 대응:** Task 1이 anchor/IoU/box encoding, Task 2가 RPNHead, Task 3이 multi-task loss를 맡는다.

**입력과 출력 shape:** 입력 [B,1,16,16] → feature [B,12,4,4] → object [B,16], delta [B,16,4].

**포트폴리오 구현 범위:** 두 번째 Fast R-CNN stage와 NMS는 제외하고 논문의 RPN 학습 경로를 깊게 구현한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Fig. 3: sliding RPN
- **노트북에서 구현할 부분:** Task 1 anchor grid
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 16 anchors

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 1: config, anchor grid, IoU, box delta encoding을 구현하세요.
@dataclass
class RPNConfig:
    pass


def generate_anchors(grid_size, box_size, device):
    raise NotImplementedError


def pairwise_iou(first, second):
    raise NotImplementedError


def encode_box(anchor, target):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.1: anchors
- **노트북에서 구현할 부분:** Task 1 IoU
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** best anchor

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 2: shared feature와 두 sibling head를 갖는 RegionProposalNetwork를 구현하세요.
class RegionProposalNetwork(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward(self, inputs):
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.2 Eq. (2): box parameterization
- **노트북에서 구현할 부분:** Task 1 delta
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** identity delta

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 3: anchor target, RPN loss, train/evaluate 흐름을 구현하세요.
def build_rpn_targets(anchors, target_boxes):
    raise NotImplementedError


def rpn_loss(prediction, targets, config):
    raise NotImplementedError


def train_rpn_step(model, optimizer, inputs, targets, config):
    raise NotImplementedError


def evaluate_rpn(model, inputs, object_targets):
    """가장 높은 objectness를 갖는 anchor의 recall을 반환한다."""
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.